# U-Net on your new dataset

Select the project `.venv` kernel and run all cells. Images are in `data/936/data`; same-name masks are in `data/936/mask`. Defaults: images 1Ã¢â‚¬â€œ5, up to 120 epochs, RGB input, three-class softmax, 128x128 resolution, 16 base filters. Class IDs are preserved as 0, 1, and 2; semantic names are not assumed.

`LIMIT=5` uses the same images for training and evaluation, so metrics are a sanity check. Set `LIMIT=None` for the full dataset with a separate held-out test split.

The multiclass head uses all decoder features. For this tiny sanity run, dropout is disabled, the initial learning rate is 0.001, and training runs for 120 epochs. Outputs are model predictions, not altered ground-truth masks.

Run `dataPrepare.ipynb` first to inspect pairs and save the preparation manifest. This notebook imports shared helpers from `data.py` and orchestration from `main.py`; `main.py` builds the U-Net from `model.py`. If preparation was skipped, `prepare_run` validates the same dataset automatically.


In [ ]:
import json, os, sys
from pathlib import Path
from types import SimpleNamespace
from datetime import datetime
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np
from PIL import Image
import tensorflow as tf
import matplotlib.pyplot as plt
from data import make_dataset, plot_predictions
from main import prepare_run, train_model, evaluate_model

# Set LIMIT=None for all 936 pairs with a held-out test split.
DATASET = Path('data/936')
LIMIT = 5
EPOCHS = 120
IMAGE_SIZE = 128
BATCH_SIZE = 4
BASE_FILTERS = 16
LEARNING_RATE = 1e-3
DROPOUT_RATE = 0.0 if LIMIT is not None else 0.5
OUTPUT = Path('runs') / ('notebook_five_images_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
args = SimpleNamespace(dataset=DATASET, limit=LIMIT, epochs=EPOCHS,
    size=IMAGE_SIZE, batch_size=BATCH_SIZE, base_filters=BASE_FILTERS, learning_rate=LEARNING_RATE, dropout_rate=DROPOUT_RATE, output=OUTPUT)
if IMAGE_SIZE < 16 or IMAGE_SIZE % 16:
    raise ValueError('IMAGE_SIZE must be a positive multiple of 16')
if min(EPOCHS, BATCH_SIZE, BASE_FILTERS) < 1 or (LIMIT is not None and LIMIT < 1):
    raise ValueError('All parameters must be positive')
print('Python:', sys.executable)
print('TensorFlow:', tf.__version__)
print('Dataset:', DATASET.resolve())


## Inspect and validate image/mask pairs


In [ ]:
images, masks, manifest, settings = prepare_run(args)
print('Selected pairs:', manifest)
print('Used dataPrepare.ipynb manifest:', settings['prepared_manifest_used'])


## Load and preprocess images and indexed masks


In [ ]:
datasets = {name: make_dataset(images, masks, ids, size=args.size,
    batch_size=args.batch_size, training=name == 'train') for name, ids in manifest.items()}
train, val, test = datasets['train'], datasets['validation'], datasets['test']
print('Dataset batches:', {name: len(ds) for name, ds in datasets.items()})


## Train U-Net


In [ ]:
model, history = train_model(args, train, val)


## Evaluate and save model predictions


In [ ]:
model, metrics, pred_dir = evaluate_model(args, test, manifest, settings)


## Compare images, ground truth, and predictions
This five-image run is intentionally trained longer to check whether the model can fit the samples; same-sample metrics do not measure unseen-image performance.


In [ ]:
fig = plot_predictions(images, masks, manifest['test'], pred_dir, args.output)
plt.show()
print('Artifacts:', args.output.resolve())
